#### Installation and Setup

In [ ]:
# Install Polars using Bash shell or power shell
pip install polars
 
# Install with all optional extras (Excel, timezone, plot support)
pip install 'polars[all]'
 
# Specific extras
pip install 'polars[xlsx2csv]'          # Excel reading
pip install 'polars[connectorx]'        # SQL database reading
pip install 'polars[pyarrow]'           # PyArrow integration
 
# Verify installation
import polars as pl
print(pl.__version__)                   # e.g., 0.20.x
print(pl.threadpool_size())             # Number of CPU threads Polars uses


In [ ]:
#These are for Jupyter notebook
%pip install polars
%pip install "polars[all]"
%pip install "polars[xlsx2csv]"
%pip install "polars[connectorx]"
%pip install "polars[pyarrow]"

In [ ]:
import polars as pl
print(pl.__version__)
print(pl.thread_pool_size())


1.32.3
8


In [ ]:
                    Eager Execution                                   	Lazy Execution
How it works	Runs immediately, returns DataFrame	            Builds a query plan, runs only on .collect()
API         	pl.read_csv(), df.filter(), df.group_by()	    pl.scan_csv(), lf.filter(), lf.group_by()
Memory	        Full data in memory	                            Reads only what is needed (projection/predicate pushdown)
When to use  	Interactive EDA, small data, debugging       	Large files, pipelines, production ETL
Optimization  	None                                         	Automatic query plan optimization
Output         	DataFrame                                   	LazyFrame (call .collect() to execute)


### Core Concepts
#### DataFrames and Series in Polars

Like Pandas, Polars has DataFrame (2D table) and Series (1D column) as its primary data structures. 

The key difference is that Polars DataFrames have NO row index — every row is identified purely by its position. 

This is a deliberate design choice that eliminates a major source of confusion in Pandas.


Creating DataFrames
From Python Data Structures

In [3]:
# polars From dictionary
pl_df1 = pl.DataFrame({
    "name": ["Alice", "Bob", "Charan","Ashwin"],
    "age": [25, 30, 35, None],
    "city": ["New ork", "London", "India","India"]
})
print(pl_df1)

NameError: name 'pl' is not defined

In [1]:
import pandas as pd
# pandas From dictionary
pd_df1 = pd.DataFrame({
    "name": ["Alice", "Bob", "Charan","Ashwin"],
    "age": [25, 30, 35,None],
    "city": ["New ork", "London", "India","India"]
})
print(pd_df1)

     name   age     city
0   Alice  25.0  New ork
1     Bob  30.0   London
2  Charan  35.0    India
3  Ashwin   NaN    India


In [2]:
# From list of dictionaries
data = [
    {"name": "Alice", "age": 25, "city": "New York"},
    {"name": "Bob", "age": 30, "city": "London"},
    {"name": "Charlie", "age": 35, "city": "Tokyo"}
]
df2 = pl.DataFrame(data)
print(df2)

NameError: name 'pl' is not defined

In [ ]:
# From lists
df3 = pl.DataFrame({
    "x": [1, 2, 3, 4, 5],
    "y": [2, 4, 6, 8, 10],
    "z": ["a", "b", "c", "d", "e"]
})
print(df3.head(4))

In [ ]:
# Polars uses method chaining extensively
# Create a DataFrame with columns 'a' and 'b'
df = (
    pl.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
    # Filter rows where column 'a' is greater than 1
    .filter(pl.col("a") > 1)
    # Select and transform column 'a' by multiplying by 2
    .select(pl.col("a") * 2)
)
print (df)

In [18]:
# From Files

# CSV
df_csv = pl.read_csv("customers.csv")
# JSON
df_json = pl.read_json("data.json")
# Parquet (recommended for performance)
# df_parquet = pl.read_parquet("data.parquet")
# Excel
df_excel = pl.read_excel("../sales_report.xlsx")
# With options
df_csv_opts = pl.read_csv(
    "customers.csv",
    separator=",",
    has_header=True,
    ignore_errors=True,
    null_values=["", "NULL", "nan"]
)


In [ ]:
print(df_csv)



In [ ]:
print(df_json)  

In [ ]:
print(df_excel)

In [ ]:
print(df_csv_opts)

In [ ]:
# Creating Test Data

# Random data for practice
import random
import string
from datetime import date
n_rows = 1000
df_test = pl.DataFrame({
    "id": range(n_rows),
    "category": [random.choice(["A", "B", "C", "D"]) for _ in range(n_rows)],
    "value": [random.uniform(0, 100) for _ in range(n_rows)],
    "date": [date(2025, random.randint(1, 12), random.randint(1, 28)) for _ in range(n_rows)],
    "flag": [random.choice([True, False]) for _ in range(n_rows)]
})
print(df_test.head(10))

In [ ]:
# Basic Operations
# Viewing Data

# First/last rows
print(df_test.head())
print(df_test.tail(3))
# Shape and info
print(f"Shape: {df_test.shape}")
print(f"Columns: {df_test.columns}")
print(f"Data types: {df_test.dtypes}")



In [ ]:
# Summary statistics
print(df_test.describe())
# Unique values
print(df_test["category"].value_counts())

#### Column Operation

In [ ]:
# Select columns
df_subset = df_test.select(["id", "category", "value"])
print(df_subset.head())

In [ ]:
# Select with expressions
df_expr = df_test.select([
    pl.col("id"),
    pl.col("value").round(2).alias("rounded_value"),
    pl.col("category").str.to_uppercase().alias("upper_category")
])

print(df_expr.head())

In [ ]:
# Add columns
df_with_new = df_test.with_columns(
    (pl.col("value") * 2).alias("doubled_value"),
    pl.col("date").dt.year().alias("year"),
    pl.when(pl.col("value") > 50).then(pl.lit("High")).otherwise(pl.lit("Low")).alias("level")
)

print(df_with_new.head())

In [ ]:
# Drop columns
df_dropped = df_test.drop(["flag"])
print(df_dropped.head())

#### Data Types

In [ ]:
# Check types
print(df_test.schema)
# Cast types
df_cast = df_test.with_columns([
    pl.col("id").cast(pl.Utf8).alias("id_str"),
    pl.col("value").cast(pl.Int32).alias("value_int")
])

print(df_cast.head())

In [ ]:
# Working with nulls
df_with_nulls = pl.DataFrame({
    "a": [1, 2, None, 4],
    "b": [None, "hello", "world", None]
})

print(df_with_nulls.head())

In [ ]:
# Handle nulls
df_clean = df_with_nulls.with_columns([
    pl.col("a").fill_null(0),
    pl.col("b").fill_null("missing")
])

print(df_clean.head())

#### Data Selection & Filtering
#### Filtering Rows

In [ ]:
df_test

In [ ]:
# Simple filter
high_value = df_test.filter(pl.col("value") > 80)
print(high_value.head())

In [ ]:
# Multiple conditions
complex_filter = df_test.filter(
    (pl.col("value") > 50) & 
    (pl.col("category").is_in(["A", "B"])) &
    (pl.col("flag") == True)
)

print(complex_filter.head())

In [ ]:
# String operations
text_filter = df_test.filter(
    pl.col("category").str.contains("A")
)
print(text_filter.head())

In [ ]:
# Date filtering
date_filter = df_test.filter(
    pl.col("date") > date(2025, 6, 1)
)
print(date_filter.head())

#### Advanced Selection

In [ ]:
# Select by data type
numeric_cols = df_test.select(pl.col(pl.NUMERIC_DTYPES))
# Select with regex
id_cols = df_test.select(pl.col("^.*id$"))  # columns ending with "_id"
print(numeric_cols.head())
print(id_cols.head())

In [ ]:
# Conditional selection
conditional = df_test.select([
    pl.col("id"),
    pl.when(pl.col("value") > 75)
    .then(pl.col("value"))
    .otherwise(None)
    .alias("high_values_only")
])


print(conditional.head(10))


In [ ]:
# Select and transform
transformed = df_test.select([
    pl.col("id"),
    pl.col("value").map_elements(lambda x: "high" if x > 50 else "low").alias("category_new")
])
print(transformed.head())

### Data Transformation
#### Sorting

In [ ]:
# Single column
sorted_df = df_test.sort("value", descending=True)
print(sorted_df.head())
# Multiple columns
multi_sorted = df_test.sort(["category", "value"], descending=[False, True])
print(multi_sorted.head())
# With nulls handling
sorted_nulls = df_with_nulls.sort("a", nulls_last=True)
print(sorted_nulls.head())


### Summary
Contexts — select(), filter(), with_columns(), agg()
A context is a DataFrame method that accepts and evaluates expressions. Different contexts have different meanings for the same expression.

Context	Method	Returns	Use Case
Select	.select()	New DataFrame with selected/computed cols	Choose or transform columns
Filter	.filter()	Subset of rows	Row-level conditions
Add/Modify	.with_columns()	Same shape with added/updated columns	Add computed columns
Aggregate	.group_by().agg()	One row per group	Summaries by category

### Self study 
#### Try it your self

In [48]:

import polars as pl
df = pl.DataFrame({
    'region': ['North','South','North','East','South'],
    'product': ['A','B','A','C','A'],
    'sales'  : [100, 200, 150, 300, 250],
    'units'  : [10, 20, 15, 30, 25],
})
 

In [ ]:
# select() — pick and compute columns (drops others)
print(df.select([
    pl.col('region'),
    pl.col('sales'),
    (pl.col('sales') / pl.col('units')).alias('avg_price'),
]))
   

In [ ]:
# filter() — keep rows matching condition
print(df.filter(pl.col('sales') > 150))
 
# with_columns() — add/replace columns, keeps all original cols
df.with_columns([
    (pl.col('sales') * 1.1).alias('sales_adjusted'),
    pl.col('product').str.to_uppercase().alias('product_upper'),
])
 

In [ ]:
# group_by().agg() — aggregate per group
df.group_by('region').agg([
    pl.col('sales').sum().alias('total_sales'),
    pl.col('sales').mean().alias('avg_sales'),
    pl.col('units').sum().alias('total_units'),
    pl.col('product').n_unique().alias('unique_products'),
])
 


In [ ]:
#Column Operations and Aliases
 
# alias() — rename the result of an expression
pl.col('salary').mean().alias('mean_salary')
 
# Operate on multiple columns at once
pl.col(['sales', 'units']).sum()          # Sum of both columns
 
# Map a function across a column
pl.col('name').map_elements(lambda x: x.split()[0], return_dtype=pl.Utf8).alias('first_name')
 
# Null-safe operations
pl.col('salary').fill_null(strategy='mean')     # Fill nulls with column mean
pl.col('salary').fill_null(pl.col('salary').median())
 
# Sorting within expression
pl.col('sales').sort_by(pl.col('date'))
 


In [ ]:
# String Operations

text_df = pl.DataFrame({
    "text": ["Hello World", "POLARS rocks", "Data Science"]
})
string_ops = text_df.with_columns([
    pl.col("text").str.to_lowercase().alias("lower"),
    pl.col("text").str.to_uppercase().alias("upper"),
    pl.col("text").str.len().alias("length"),
    pl.col("text").str.contains("o").alias("has_o"),
    pl.col("text").str.replace("o", "0").alias("replaced"),
    pl.col("text").str.split(" ").alias("split")
])


#### Aggregations & Grouping
#### Basic Aggregations

In [ ]:
# Single aggregation
mean_value = df_test.select(pl.col("value").mean())
total_records = df_test.select(pl.count())
# Multiple aggregations
stats = df_test.select([
    pl.col("value").mean().alias("mean_value"),
    pl.col("value").median().alias("median_value"),
    pl.col("value").std().alias("std_value"),
    pl.col("value").min().alias("min_value"),
    pl.col("value").max().alias("max_value")
])


#### GroupBy Operations

In [ ]:
# Simple groupby
category_stats = df_test.group_by("category").agg([
    pl.col("value").mean().alias("avg_value"),
    pl.col("value").count().alias("count"),
    pl.col("value").sum().alias("total_value")
])

print(category_stats)

In [ ]:
# Multiple grouping columns
multi_group = df_test.group_by(["category", "flag"]).agg([
    pl.col("value").mean().alias("avg_value"),
    pl.col("id").count().alias("count")
])
print(multi_group)

In [ ]:
# Advanced aggregations
advanced_agg = df_test.group_by("category").agg([
    pl.col("value").quantile(0.25).alias("q25"),
    pl.col("value").quantile(0.75).alias("q75"),
    pl.col("value").filter(pl.col("value") > 50).mean().alias("high_avg"),
    pl.col("date").min().alias("earliest_date"),
    pl.col("date").max().alias("latest_date")
])
print(advanced_agg)

### Day - 2

#### Data I/O  
Polars provides fast, memory-efficient I/O for all major data formats. Every eager read function has a lazy scan counterpart. Understanding when to use each is critical for production data pipelines.

#### Reading CSV
pl.read_csv() — Eager (loads entire file)


In [ ]:
import polars as pl

df = pl.read_csv("sales.csv")

print(df.schema)
print(df.shape)
print(df.head(5))


In [ ]:
import polars as pl

df = pl.read_csv(
    "sales.csv",
    separator=",",                   # CSV delimiter
    has_header=True,                 # first row is header in the file
    null_values=["N/A", "NULL", "--"],
    infer_schema_length=1000,        # rows to infer schema from (increase if needed)
    encoding="utf8",
    comment_prefix="#",              # ignore commented lines, if any
    schema_overrides={               # force dtypes where it matters
        "order_id": pl.Utf8,
        "product_name": pl.Utf8,
        "category": pl.Utf8,
        "amount": pl.Int64,          # values look like integers in the file
        "date": pl.Date,             # parse yyyy-mm-dd
        "year": pl.Int32,
        "region": pl.Utf8,
        "customer_id": pl.Utf8,
    },
    try_parse_dates=True,            # will parse date-like columns automatically
)

print(df.schema)     # Column name → dtype mapping
print(df.shape)      # (rows, cols)
print(df.head(5))

#### pl.scan_csv() — Lazy (best for large files)
Here’s the lazy Polars (pl.scan_csv) version rewritten to match your sales_data.csv columns and data. The file has these headers: date, product, quantity, price, region (no id column), and it contains some missing values in quantity and region. [sales_data | Excel]

In [ ]:
import polars as pl

# scan_csv creates a LazyFrame — nothing is read yet
lf = pl.scan_csv(
    "sales_data.csv",
    try_parse_dates=True,  # 'date' is in YYYY-MM-DD format
    null_values=["", "NULL", "N/A", "--"],  # treat empty region/quantity as null
)

# Build a query (still not executed)
query = (
    lf
    .filter(pl.col("price") > 500)  # Predicate pushdown (price exists in your file)
    .select(["date", "product", "quantity", "price", "region"])  # Projection pushdown
    .with_columns(
        (pl.col("price") * 1.10).alias("price_gst")  # add 10% GST
    )
)

# Execute the optimized query
result = query.collect()  # Executes NOW

# Inspect query plan (what Polars will do)
print(query.explain(optimized=True))

# Inspect output
print(result.head(10))

### Writing / Exporting Data

In [ ]:

import polars as pl
df = pl.DataFrame({'id':[1,2,3], 'name':['A','B','C'], 'value':[10.1, 20.2, 30.3]})
 
# ─── CSV ───────────────────────────────────────────────────────
df.write_csv('output.csv')                      # Default comma separator
df.write_csv('output.tsv', separator='\t')       # Tab-separated
df.write_csv('output.csv', null_value='NULL')    # Custom null representation
df.write_csv('output.csv.gz')                    # Auto-compressed if .gz extension
 
# ─── JSON ──────────────────────────────────────────────────────
df.write_json('output.json')                     # Compact JSON
df.write_json('output.json', pretty=True)        # Human-readable
df.write_ndjson('output.ndjson')                 # Newline-delimited (streaming-safe)
 
# ─── Excel ─────────────────────────────────────────────────────
# pip install 'polars[xlsx2csv]' and openpyxl
df.write_excel('output.xlsx', worksheet='Results')
 



### Conclusion & Next Steps
#### Key Takeaways
    1. Polars is fast and memory-efficient - Use it for large datasets
    2. Lazy evaluation - Use scan_* and LazyFrame for better performance
    3. Method chaining - Build complex queries step by step
    4. Expression system - pl.col() is your friend for column operations

Type awareness - Polars has a rich type system, use it

#### Resources
    • Official Documentation: https://pola-rs.github.io/polars/
    • GitHub Repository: https://github.com/pola-rs/polars
    • Community: Polars Discord server
    * Performance comparisons with pandas online